# Clean the dataset

Runs the cleaning steps on the raw PhiUSIIL CSV, in order, and prints how many rows each step removes. Every step reads from the raw file, so running the notebook twice gives the same counts.

| Step | Rule |
|---|---|
| 1. Duplicate URLs | A row whose `URL` exactly matches another row's is a duplicate, and the first copy is kept. If copies of a URL have different labels, every copy is removed and counted separately. |
| 2. Missing, empty and out-of-range values | Out of range means a count below 0, a ratio or rate outside 0 to 1, a score outside 0 to 100, a 0/1 column or the label holding anything else. The `Title` placeholder `0` is blanked. Every other problem removes its row. |
| 3. Recode the label | New label = 1 − raw label, so **1 = phishing** and **0 = legitimate**. The raw file uses the reverse. |

The CSV is read so that only truly empty cells count as missing (`keep_default_na=False`). pandas would otherwise turn text such as `NA` or `null` into missing values. The file uses `na` for Namibia's real domain, `.na`, which must stay as text.

The steps live in `ml/cleaning/` and are tested in `ml/tests/`. `run_cleaning()` runs all three in order from the raw data, and this notebook only calls it and explains the results. Deduplication happens here, before any train/test split.

The last section writes the cleaning report to `ml/reports/cleaning_report.md`, after running the cleaning a second time to check it gives the same numbers.

Get the CSV first by following `ml/README.md`. Links in the data are only ever handled as text.

In [1]:
import sys
from pathlib import Path

import pandas as pd

CSV_NAME = "PhiUSIIL_Phishing_URL_Dataset.csv"


def find_ml_dir(start: Path) -> Path:
    """Walk up to the repo root (the folder holding AGENTS.md), so the notebook runs from any working directory inside the repo."""
    for folder in [start, *start.parents]:
        if (folder / "AGENTS.md").exists() and (folder / "ml").is_dir():
            return folder / "ml"
    raise FileNotFoundError("Could not find the ml/ folder. Open this notebook from inside the LinkGuard repo.")


ML_DIR = find_ml_dir(Path.cwd().resolve())
DATA_PATH = ML_DIR / "data" / CSV_NAME
sys.path.insert(0, str(ML_DIR))

from cleaning.labels import LEGITIMATE, PHISHING  # noqa: E402
from cleaning.report import run_cleaning, steps_table  # noqa: E402

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download it by following 'Get the dataset' in ml/README.md, then run this notebook again."
    )

raw = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""])
print(f"Raw file: {len(raw):,} rows")

Raw file: 235,795 rows


## Step 1: duplicate URLs

In [2]:
run = run_cleaning(raw)  # runs steps 1 to 3; the cells below look at each step's result
duplicates = run.duplicates

print(f"Rows before:                    {duplicates.rows_before:,}")
print(f"Extra copies removed:           {duplicates.duplicate_rows_removed:,}")
print(f"URLs with both labels:          {duplicates.conflicting_urls:,}")
print(f"  rows removed for that reason: {duplicates.conflicting_rows_removed:,}")
print(f"Rows after:                     {duplicates.rows_after:,}")

Rows before:                    235,795
Extra copies removed:           425
URLs with both labels:          0
  rows removed for that reason: 0
Rows after:                     235,370


## Why this is not 808

Other analyses report about 808 duplicates. The difference comes from what counts as a duplicate:

- **808** is the number of rows whose **numeric feature values** exactly match an earlier row's, even though their URLs are different.
- **This step** follows the story's spec: a duplicate is a row with the **same `URL`**.

The cell below works out both numbers from the raw file.

In [3]:
same_url = raw.duplicated("URL").sum()
same_numeric_features = raw.select_dtypes("number").duplicated().sum()

print(f"Extra rows with the same URL:                 {same_url:,}")
print(f"Extra rows with the same numeric features:    {same_numeric_features:,}")

Extra rows with the same URL:                 425
Extra rows with the same numeric features:    808


## What the duplicates look like

The cell below shows how many copies each duplicated URL has, which label they carry, and how often the two copies' features disagree.

In [4]:
copies = raw[raw.duplicated("URL", keep=False)]
copies_per_url = copies.groupby("URL").size()

print(f"Duplicated URLs: {copies_per_url.size:,}")
print(f"Copies per duplicated URL: {copies_per_url.value_counts().sort_index().to_dict()}  (copies: number of URLs)")
print(f"Raw label of their copies: {copies['label'].value_counts().to_dict()}  (raw file: 0 = phishing)")

feature_columns = [column for column in raw.columns if column not in ("FILENAME", "URL", "label")]
differs = copies.groupby("URL")[feature_columns].nunique().gt(1)
print(f"URLs whose two copies differ in at least one feature: {differs.any(axis=1).sum():,} of {copies_per_url.size:,}")
print(f"URLs whose copies even differ in URLLength:           {differs['URLLength'].sum():,}")

Duplicated URLs: 425
Copies per duplicated URL: {2: 425}  (copies: number of URLs)
Raw label of their copies: {0: 850}  (raw file: 0 = phishing)
URLs whose two copies differ in at least one feature: 425 of 425
URLs whose copies even differ in URLLength:           425


Every duplicated URL appears exactly twice, and both copies are labelled phishing. The two copies are never identical rows. Every pair differs in its page features, and even in `URLLength`, although the URL text is exactly the same. So the feature values weren't worked out from the URL text alone, and each copy is probably a separate crawl of the same link.

The spec's rule is to keep the first copy. Both copies share a label, so the choice doesn't change the label balance. It does mean the kept row's features come from one crawl only.

## Step 2: missing, empty and out-of-range values

Runs on the output of step 1. The first table counts problems in every column. The second lists only the columns that have a problem, with the action taken.

In [5]:
values = run.values

problems = values.problems
print(f"Checked {len(problems)} columns. Problems per column:\n")
print(problems.to_string())

Checked 56 columns. Problems per column:

                            missing  empty  out_of_range
FILENAME                          0      0             0
URL                               0      0             0
URLLength                         0      0             0
Domain                            0      0             0
DomainLength                      0      0             0
IsDomainIP                        0      0             0
TLD                               0      0             0
URLSimilarityIndex                0      0             0
CharContinuationRate              0      0             0
TLDLegitimateProb                 0      0             0
URLCharProb                       0      0             0
TLDLength                         0      0             0
NoOfSubDomain                     0      0             0
HasObfuscation                    0      0             0
NoOfObfuscatedChar                0      0             0
ObfuscationRatio                  0      0    

In [6]:
ACTIONS = {
    ("Title", "empty"): "The text '0' marks a page with no title (HasTitle = 0). Fixed: the title is blanked and the row kept.",
}
DEFAULT_ACTION = "Row removed: the model can't use a row with a broken feature or label."

found = problems.stack()
found = found[found > 0]
actions = pd.DataFrame(
    [
        {"Column": column, "Problem": kind, "Rows": int(count), "Action": ACTIONS.get((column, kind), DEFAULT_ACTION)}
        for (column, kind), count in found.items()
    ],
    columns=["Column", "Problem", "Rows", "Action"],
)
print(actions.to_string(index=False) if len(actions) else "No problems found.")
print(f"\nTitles fixed: {values.titles_fixed:,}")
print(f"Rows removed: {values.rows_removed:,}")
print(f"Rows after:   {values.rows_after:,}")

Column Problem  Rows                                                                                                Action
 Title   empty 32612 The text '0' marks a page with no title (HasTitle = 0). Fixed: the title is blanked and the row kept.

Titles fixed: 32,612
Rows removed: 0
Rows after:   235,370


By the story's rules the file is in good shape. There are no missing cells, no negative counts, every ratio and rate is within 0 to 1, every score is within 0 to 100, and every 0/1 column and the label hold only 0 or 1. The one problem is the `Title` placeholder. Pages with no title store the text `0` as their title, which a text feature would treat as a real word. It is blanked, so `HasTitle = 0` and an empty title now agree.

A few pages that do have a title (`HasTitle = 1`) are literally titled `0`. Those are kept as they are.

## Step 3: recode the label

The raw file codes legitimate as 1 and phishing as 0. Every LinkGuard model uses the opposite, so phishing, the class we warn about, is 1. The check below compares each cleaned row with the same row in the raw file. The cleaning steps keep the raw row numbers, so rows can be matched exactly.

In [7]:
labels = run.labels
cleaned = run.cleaned

raw_label = raw.loc[cleaned.index, "label"]
was_phishing = raw_label == 0
was_legitimate = raw_label == 1
flipped_correctly = (cleaned.loc[was_phishing, "label"] == PHISHING).all() and (
    cleaned.loc[was_legitimate, "label"] == LEGITIMATE
).all()
if not flipped_correctly:
    raise ValueError("Label recode check failed: some rows were not flipped.")

print(f"Raw 0 (phishing)   -> now {PHISHING}: {int(was_phishing.sum()):,} rows")
print(f"Raw 1 (legitimate) -> now {LEGITIMATE}: {int(was_legitimate.sum()):,} rows")
print("Every row checked against the raw file: flipped correctly.\n")
print("Label counts after the recode:")
print(f"  1 (phishing):   {labels.phishing:,}")
print(f"  0 (legitimate): {labels.legitimate:,}")

Raw 0 (phishing)   -> now 1: 100,520 rows
Raw 1 (legitimate) -> now 0: 134,850 rows
Every row checked against the raw file: flipped correctly.

Label counts after the recode:
  1 (phishing):   100,520
  0 (legitimate): 134,850


## Cleaning report

### Rows at each step

Starts from the raw file. Each step's "rows before" must equal the previous step's "rows after", or `steps_table()` stops with an error.

In [8]:
steps = steps_table(run.steps)
if steps.iloc[0]["Rows before"] != len(raw):
    raise ValueError(f"The first step should start from all {len(raw):,} raw rows.")
print(steps.to_string(index=False))

                                     Step  Rows before  Rows after  Rows removed
                        1. Duplicate URLs       235795      235370           425
2. Missing, empty and out-of-range values       235370      235370             0
                      3. Recode the label       235370      235370             0
                                    Total       235795      235370           425


### Running it again gives the same numbers

The cleaning runs a second time from the raw file, and the two runs must agree on every count and on the cleaned data itself.

In [9]:
second_run = run_cleaning(raw)

same_counts = second_run.steps == run.steps
same_data = second_run.cleaned.equals(run.cleaned)
if not (same_counts and same_data):
    raise ValueError("A second run of the cleaning gave different results.")
print("Second run from the raw file: same row counts and identical cleaned data.")

Second run from the raw file: same row counts and identical cleaned data.


### Save the report

Writes the numbers above to `ml/reports/cleaning_report.md`, so the report always matches a real run.

In [10]:
def to_markdown_table(table: pd.DataFrame) -> str:
    header = "| " + " | ".join(table.columns) + " |"
    divider = "|" + "|".join("---" for _ in table.columns) + "|"
    body = ["| " + " | ".join(f"{cell:,}" if pd.api.types.is_integer(cell) else str(cell) for cell in row) + " |"
            for row in table.itertuples(index=False)]
    return "\n".join([header, divider, *body])


missing_found = int(values.problems["missing"].sum())
out_of_range_found = int(values.problems["out_of_range"].sum())
namibia_domains = int((raw["TLD"] == "na").sum())
report = f"""# PhiUSIIL cleaning report

Generated by `ml/notebooks/03_clean_dataset.ipynb` from the raw file `{CSV_NAME}` ({len(raw):,} rows, {raw.shape[1]} columns).
The steps are in `ml/cleaning/`, run in order by `run_cleaning()`.

## Rows at each step

{to_markdown_table(steps)}

Running the cleaning a second time from the raw file gives the same counts and identical cleaned data.

## What each step did

### 1. Duplicate URLs

- A duplicate is a row whose `URL` exactly matches another row's, and the first copy is kept.
- **{duplicates.duplicate_rows_removed:,} extra copies removed.** {duplicates.conflicting_urls:,} URLs had copies with different labels ({duplicates.conflicting_rows_removed:,} rows removed for that reason).
- **Compared with other analyses' ~808:** that figure counts rows whose numeric feature values match an earlier row's ({same_numeric_features:,} in the raw file), even though their URLs differ. Rows with the same URL number {same_url:,}. The 808 lookalike rows are kept for now and are being checked for train/test leakage separately.
- Every duplicated URL appears exactly twice, both copies are labelled phishing, and every pair has different page features, so they look like two separate crawls of the same link.

### 2. Missing, empty and out-of-range values

- Out of range means a count below 0, a ratio or rate outside 0 to 1, a score outside 0 to 100, or a 0/1 column or label holding anything else.
- Missing cells found: **{missing_found:,}**. Out-of-range values found: **{out_of_range_found:,}**.
- **`Title`:** {values.titles_fixed:,} pages with no title (`HasTitle = 0`) store the text `0` as their title. **Action: fixed.** The title is blanked and the row kept.
- **{values.rows_removed:,} rows removed.**
- The CSV is read with `keep_default_na=False`, so the {namibia_domains:,} `.na` (Namibia) domains stay as text instead of becoming missing values.

### 3. Recode the label

- **The label is inverted.** The raw file uses 1 = legitimate and 0 = phishing. Cleaning sets new label = 1 − raw label, so **1 = phishing** and **0 = legitimate**.
- Every cleaned row was checked against the same row in the raw file and was flipped correctly.
- Label counts after cleaning: **1 (phishing): {labels.phishing:,}**, **0 (legitimate): {labels.legitimate:,}**.

## Result

{len(raw):,} raw rows → **{len(cleaned):,} cleaned rows**, {len(raw) - len(cleaned):,} removed in total. Every column is kept.
"""

REPORT_PATH = ML_DIR / "reports" / "cleaning_report.md"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Saved {REPORT_PATH.relative_to(ML_DIR.parent)}")

Saved ml/reports/cleaning_report.md
